# 🦉 Notebook 1: Stack Overflow — Class Design

**Stack Overflow** is a Q&A site where users ask questions, post answers, vote, earn **reputation**,
and unlock privileges with **badges**. It's a popular **object-oriented design (OOD)** interview question.

In a real interview, the grade comes *less* from "did your code run" and *more* from how you:

1. Clarify the requirements (ask questions!).
2. Identify **actors** and **use-cases**.
3. Pick out **entities** (nouns) and **behaviors** (verbs).
4. Draw how classes relate.
5. Sanity-check against **SOLID** principles.

This notebook walks those steps. Notebook 2 turns the design into runnable code
(bad → good → best). Notebook 3 adds real-world extensions.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/stack-overflow
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. Clarifying questions

Always start by asking. A few good ones for Stack Overflow:

- **Users**: are there *guests* (read-only) vs *members* (post/vote) vs *moderators* (close/reopen) vs *admins* (ban)?
- **Posts**: can both questions and answers receive **comments**? How long are they?
- **Voting**: can a user **change** their vote? **Remove** it? Vote on their own posts (usually *no*)?
- **Reputation**: how many points per up-vote on a question vs an answer? Down-vote penalty?
- **Accepted answers**: can the asker accept multiple answers, or only one?
- **Tags**: free-form or curated? Is there a limit per question?
- **Badges**: automatic (based on reputation milestones) or manually awarded?
- **Search**: by text? by tag? sorted how? (active / newest / votes)
- **Moderation**: who can close/reopen/delete questions? What are the reasons?
- **Scale**: is this a single-process exercise, or do we need to think about DBs and sharding? *(For an OOD interview: start single-process.)*

> ⚖️ **Rule of thumb:** it's better to ask three questions and design the right thing,
> than to race ahead and design for assumptions the interviewer didn't make.


## 2. Actors and use-cases

An **actor** is anyone (or anything) that interacts with the system.

| Actor      | What they do |
|------------|--------------|
| Guest      | Search and read questions. Can't post or vote. |
| Member     | Everything a guest can do, plus: post questions/answers, add comments, vote, accept answers on their own questions. |
| Moderator  | Everything a member can do, plus: close/reopen/delete any question. |
| Admin      | Block or unblock members. |
| System     | Awards badges, sends notifications. |

### Core use-cases (happy path)

1. `ask_question(title, body, tags)` → returns a `Question`.
2. `post_answer(question, body)` → returns an `Answer`.
3. `vote(post, UP | DOWN)` → updates the post's score and the author's reputation.
4. `accept_answer(question, answer)` → only the asker; only one accepted answer.
5. `comment(post, text)` → short free-form note attached to a Question or Answer.
6. `search(text=..., tag=...)` → list of questions.
7. `close_question(question, reason)` → moderator action.


## 3. Entities (the "nouns")

Nouns from the requirements become classes:

- **User** — `id`, `name`, `reputation`, `badges`.
- **Post** — abstract base class for things that can be voted on and commented on. Has `body`, `author`, `votes`, `comments`.
- **Question** *(a Post)* — adds `title`, `tags`, `answers`, `accepted_answer`, `status` (open / closed / deleted).
- **Answer** *(a Post)* — adds a back-reference to its `Question`.
- **Comment** — short text attached to any Post.
- **Vote** — represented as a `user_id -> UP/DOWN` map on each Post (simpler than a full class for an intro lab).
- **Tag** — for this lab, just a string. A real system would make `Tag` its own entity with description, follower-count, etc.
- **Badge** — name + rule (e.g., *"Nice Answer" = any answer reaches score 10*).

### "Is-a" vs "has-a"

- `Question` **is-a** `Post` ➜ inheritance.
- `Answer` **is-a** `Post` ➜ inheritance.
- `Question` **has-a** list of `Answer` ➜ composition.
- `Question` and `Answer` **have-many** `Comment` ➜ composition.
- `User` **has-many** `Badge`s ➜ composition.


## 4. UML-ish class diagram

```
        User (1) ---authors---> (*) Post  [abstract]
                                    |
                                    +-- has many --> Comment
                                    |
                          +---------+---------+
                          |                   |
                      Question  ---(*)---> Answer
                      title, tags, status, accepted_answer
```

Key relationships:

- `User` 1..* `Post`   (a user authors many posts)
- `Post` has-many `Comment`
- `Question` 1..* `Answer`   (a question has many answers)
- `Question` 0..1 `Answer`   (the accepted answer, if any)

> 🧠 **Why the shared `Post` base class?**
> Voting, commenting, and scoring are identical for questions and answers.
> Putting that logic once on `Post` avoids duplication and respects the **DRY** principle.


## 5. Sanity-check: SOLID

| Principle | How our design honors it |
|-----------|--------------------------|
| **S**ingle responsibility | `User` tracks identity & reputation; `Post` tracks votes/comments; `Question` orchestrates answers; a `ReputationPolicy` (notebook 3) owns scoring rules. |
| **O**pen/closed | Adding a new post-like thing (e.g., `WikiPost`) doesn't require modifying `User` or `Question` — we just subclass `Post`. |
| **L**iskov substitution | Anywhere that expects a `Post`, an `Answer` or a `Question` works (same `score`, same `comment()`). |
| **I**nterface segregation | Guests don't need `vote()`; our design keeps voting as a *function*, not a method guests have to expose. |
| **D**ependency inversion | In notebook 3 we'll inject a **notifier** and a **reputation policy**, so `Question` depends on *abstractions*, not hard-coded rules. |


## 7. The design as a runnable skeleton

Diagrams are cheap. Before we commit to this model, let's write the **skeleton** —
the classes, the enums, the relationships, and *nothing else*. No storage, no search,
no badges, no moderation. Just enough code to answer one question:

> *"Does the class diagram in section 4 actually hold together?"*

This is design work, not implementation work. If the skeleton is awkward to write, the
diagram was wrong, and it is far cheaper to find that out now than after 300 lines.

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
import itertools


# ── Value types: fixed sets of values become enums, never strings ────────────
class VoteType(Enum):
    UP   = +1
    DOWN = -1

class QuestionStatus(Enum):
    OPEN    = 'open'
    CLOSED  = 'closed'
    DELETED = 'deleted'


# ── Entities ────────────────────────────────────────────────────────────────
@dataclass
class Comment:
    """Attached to any Post. Short, unvotable, immutable once written."""
    author: 'User'
    text: str


@dataclass
class Badge:
    """name + rule. The rule is data, not an `if` buried inside User."""
    name: str
    earned_by: callable          # (User) -> bool


@dataclass
class User:
    id: str
    name: str
    reputation: int = 0
    badges: list[Badge] = field(default_factory=list)


_post_ids = itertools.count(1)


class Post(ABC):
    """Everything votable and commentable.

    Question and Answer differ in *what they are*, not in *how voting works* —
    so voting, scoring and commenting live here, once. That is the DRY argument
    for the shared base class from section 4.
    """

    def __init__(self, author: User, body: str):
        self.id       = next(_post_ids)
        self.author   = author
        self.body     = body
        self.votes:    dict[str, VoteType] = {}   # one vote per user: the key IS the rule
        self.comments: list[Comment] = []

    @property
    def score(self) -> int:
        return sum(v.value for v in self.votes.values())

    def add_comment(self, author: User, text: str) -> Comment:
        c = Comment(author, text)
        self.comments.append(c)
        return c

    # The ONE thing subclasses genuinely disagree about: what an up-vote is worth.
    # Note this is a *polymorphic* answer to a question a naive design would answer
    # with `if isinstance(post, Answer)`. Whenever you catch yourself writing
    # isinstance, ask whether the subclass could just tell you the answer itself.
    @property
    @abstractmethod
    def rep_per_upvote(self) -> int: ...


class Question(Post):
    """is-a Post; has-many Answers; has 0..1 accepted Answer."""

    def __init__(self, author: User, title: str, body: str, tags: list[str]):
        super().__init__(author, body)
        self.title    = title
        self.tags     = list(tags)
        self.answers:  list[Answer] = []
        self.accepted: Answer | None = None
        self.status   = QuestionStatus.OPEN

    rep_per_upvote = 5          # concrete override of the abstract property

    def _register(self, answer: 'Answer') -> None:
        """Called by Answer.__init__ so the back-reference can never go one-way."""
        self.answers.append(answer)

    def accept(self, answer: 'Answer') -> None:
        if answer.question is not self:
            raise ValueError('that answer belongs to a different question')
        if self.accepted is not None:
            raise ValueError('only one answer can be accepted')
        self.accepted = answer


class Answer(Post):
    """is-a Post; belongs to exactly one Question (a mandatory back-reference)."""

    def __init__(self, author: User, question: Question, body: str):
        super().__init__(author, body)
        self.question = question
        question._register(self)      # the two sides are wired in ONE place

    rep_per_upvote = 10


print('skeleton defined:', [c.__name__ for c in (User, Post, Question, Answer, Comment, Badge)])

### 🧪 Does the diagram hold up?

Each assertion below is one line from the class diagram in section 4, restated as
something a computer can check. If a future edit breaks a relationship, this cell fails.

In [ ]:
def must_raise(exc, fn, *a, **kw):
    """Fail loudly when nothing is raised — a bare try/except would pass silently."""
    try:
        fn(*a, **kw)
    except exc:
        return True
    raise AssertionError(f'expected {exc.__name__}, nothing was raised')

ada, grace, bob = User('u1', 'Ada'), User('u2', 'Grace'), User('u3', 'Bob')
q  = Question(ada, 'What is OOD?', 'Explain please.', ['ood', 'design'])
a1 = Answer(grace, q, 'Start with SOLID.')
a2 = Answer(bob,   q, 'Practise with the parking-lot problem.')

# --- "Post is abstract" -----------------------------------------------------
must_raise(TypeError, Post, ada, 'body')          # you cannot instantiate the base

# --- "Question is-a Post", "Answer is-a Post" -------------------------------
assert isinstance(q, Post) and isinstance(a1, Post)
# Liskov: anything that only needs Post behaviour works for both, no isinstance.
def total_score(posts): return sum(p.score for p in posts)
assert total_score([q, a1, a2]) == 0

# --- "Question has-many Answer" (and the back-reference is symmetric) -------
assert q.answers == [a1, a2]
assert a1.question is q and a2.question is q
# You cannot end up with a half-wired pair, because Answer.__init__ does both sides.
assert all(ans in ans.question.answers for ans in q.answers)

# --- "Question has 0..1 accepted Answer" ------------------------------------
assert q.accepted is None
other_q = Question(bob, 'Other', 'body', ['x'])
stray   = Answer(ada, other_q, 'unrelated')
must_raise(ValueError, q.accept, stray)           # not an answer to THIS question
q.accept(a1)
assert q.accepted is a1
must_raise(ValueError, q.accept, a2)              # accepting is a one-way door

# --- "Post has-many Comment" (shared by both subclasses, defined once) ------
q.add_comment(bob, 'Can you narrow this down?')
a1.add_comment(ada, 'Thanks!')
assert len(q.comments) == 1 and len(a1.comments) == 1
assert q.comments[0].author is bob

# --- "one vote per user" is enforced by the data structure itself -----------
a1.votes[ada.id] = VoteType.UP
a1.votes[bob.id] = VoteType.UP
a1.votes[ada.id] = VoteType.DOWN         # Ada changes her mind — replaces, not adds
assert a1.score == 0, a1.score           # +1 (Bob) −1 (Ada)
assert len(a1.votes) == 2, 'a dict keyed by user makes double-voting impossible'

# --- reputation weight is polymorphic, not an isinstance ladder -------------
assert q.rep_per_upvote == 5 and a1.rep_per_upvote == 10
assert [p.rep_per_upvote for p in (q, a1)] == [5, 10]
# A future WikiPost subclass just declares its own rate; no existing caller changes.
class WikiPost(Post):
    rep_per_upvote = 0
w = WikiPost(ada, 'community wiki')
assert isinstance(w, Post) and w.rep_per_upvote == 0
# ...and forgetting to declare it is caught at construction time, not in production.
class Forgetful(Post): pass
must_raise(TypeError, Forgetful, ada, 'oops')

# --- Badge = name + rule (data), so new badges need no new code ------------
autobiographer = Badge('Autobiographer', earned_by=lambda u: u.reputation >= 1)
assert not autobiographer.earned_by(ada)
ada.reputation = 5
assert autobiographer.earned_by(ada)

print('✅ every relationship in the diagram holds')

### What the skeleton just taught us

Writing those ~90 lines surfaced three decisions the diagram alone left vague:

1. **`votes` is a `dict[user_id, VoteType]`, not a list.** That single choice makes
   "one vote per user" and "a user may change their vote" true *by construction* — no
   validation code required. Prefer data structures that make bad states unrepresentable
   over checks that catch them afterwards.
2. **`Answer.__init__` wires both directions of the `Question ↔ Answer` link.** If the
   caller had to remember `question.answers.append(a)`, sooner or later someone wouldn't,
   and you'd have an answer no question knows about.
3. **`rep_per_upvote` belongs on the subclass.** The obvious first draft is
   `if isinstance(post, Answer): 10 else: 5`, and it needs editing for every new post
   type. Notebook 2 deliberately shows that `isinstance` version, so you can feel the
   difference — watch for it, it's a real smell hiding inside "good" code.

Notice what's still missing on purpose: no `vote()` function, no reputation arithmetic,
no search, no persistence. A skeleton answers *"is the shape right?"* — nothing more.

## 8. What we'll build next

- **Notebook 2** — turn this design into real Python.
  We'll start with a **bad** "god class" (everything mashed into one object), then refactor
  to a **good** version (proper classes), and finish with a **best** version (enums, statuses,
  guarded self-votes, etc.).
- **Notebook 3** — real-world extensions:
  - Tags + search
  - Badges via the **observer** pattern
  - Closing / reopening questions (moderation)
  - Thread-safe voting
  - Pluggable reputation rules (**strategy** pattern)

> 💡 **Design first, implement second — but "design" is allowed to compile.** The skeleton
> above is a *thinking* tool: cheap to write, cheap to throw away, and it catches modelling
> mistakes while they still cost minutes instead of days.